# Category example structures

Visualize representative generated, matched training, and relaxed substituted training structures for the novelty categories used in `classification.ipynb`.

In [ ]:
from __future__ import annotations

import gzip
import pickle
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatviz import structure_2d

import __main__


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or shell working directory."""
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src").is_dir():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
SCRIPTS_DIR = ROOT / "scripts"
for import_path in (ROOT, NOTEBOOKS_DIR, SCRIPTS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from notebook_constants import (  # noqa: E402
    CATEGORY_LABELS,
    CATEGORY_ORDER,
    EHULL_RELAXED_FILE,
    GEN_FILE,
    METASTABLE_EHULL_MAX,
    RELAXED_SM_ANON_ENTRIES_FILE,
    RELAXED_SM_ANON_MATCHES_FILE,
    RELAXED_WYCKOFF_ENTRIES_FILE,
    RELAXED_WYCKOFF_MATCHES_FILE,
    SM_FIT_FILE,
    SMACT_VALIDITY_FILE,
    SOURCE_ORDER,
    SUBSET_OPTIONS,
    TRAIN_FILE,
)
from substitute_structures import SubstitutedEntry  # noqa: E402

from src.config import INPUT_DIR, RESULTS_DIR  # noqa: E402

# Some substituted-entry pickles were written with SubstitutedEntry resolved
# from __main__. Provide that symbol before unpickling.
__main__.SubstitutedEntry = SubstitutedEntry

ROOT, INPUT_DIR, RESULTS_DIR

In [ ]:
MODEL = "mattergen"
SUBSET = "metastable_smact_valid"
N_EXAMPLES_PER_CATEGORY = 5

if SUBSET not in SUBSET_OPTIONS:
    raise ValueError(f"SUBSET must be one of {sorted(SUBSET_OPTIONS)}, got {SUBSET!r}")
if N_EXAMPLES_PER_CATEGORY < 1:
    raise ValueError("N_EXAMPLES_PER_CATEGORY must be at least 1")

In [ ]:
def load_pickle_gz(path: Path) -> Any:
    with gzip.open(path, "rb") as file:
        return pickle.load(file)  # noqa: S301


def validate_1d_length(values: np.ndarray, expected: int, path: Path) -> np.ndarray:
    if values.ndim != 1 or len(values) != expected:
        raise ValueError(
            f"{path} has unexpected shape {values.shape}; expected ({expected},)"
        )
    return values


def load_smact_validity(path: Path, expected: int) -> np.ndarray:
    with np.load(path) as data:
        if "valid" not in data.files:
            raise ValueError(f"{path} does not contain a 'valid' array")
        values = np.asarray(data["valid"], dtype=bool)
    return validate_1d_length(values, expected, path)


def load_direct_matches(path: Path) -> pd.DataFrame:
    with np.load(path) as data:
        if "matches" not in data.files:
            raise ValueError(f"{path} does not contain a 'matches' array")
        matches = data["matches"]

    if matches.ndim != 2 or matches.shape[1] != 2:
        raise ValueError(
            f"{path} has unexpected shape {matches.shape}; expected (n, 2)"
        )
    return pd.DataFrame(matches, columns=["gen_idx", "train_idx"]).astype(int)


def load_relaxed_match_gen_indices(path: Path) -> set[int]:
    records = load_pickle_gz(path)
    return {
        int(record["gen_idx"]) for record in records if bool(record.get("match", False))
    }


def required_paths(model: str) -> dict[str, Path]:
    gen_dir = INPUT_DIR / "gen" / "preprocessed" / model
    result_dir = RESULTS_DIR / model
    train_dir = INPUT_DIR / "train" / "preprocessed"
    return {
        "generated_structures": gen_dir / GEN_FILE,
        "training_structures": train_dir / TRAIN_FILE,
        "relaxed_ehull": gen_dir / EHULL_RELAXED_FILE,
        "smact_validity": gen_dir / SMACT_VALIDITY_FILE,
        "direct_sm": result_dir / SM_FIT_FILE,
        "relaxed_sm_anon_entries": result_dir / RELAXED_SM_ANON_ENTRIES_FILE,
        "relaxed_wyckoff_entries": result_dir / RELAXED_WYCKOFF_ENTRIES_FILE,
        "relaxed_sm_anon_matches": result_dir / RELAXED_SM_ANON_MATCHES_FILE,
        "relaxed_wyckoff_matches": result_dir / RELAXED_WYCKOFF_MATCHES_FILE,
    }


paths = required_paths(MODEL)
missing_paths = pd.DataFrame(
    {"kind": kind, "path": str(path)}
    for kind, path in paths.items()
    if not path.exists()
)
if not missing_paths.empty:
    display(missing_paths)
    raise FileNotFoundError(
        f"Missing required files for MODEL={MODEL!r}. Check INPUT_DIR and RESULTS_DIR."
    )

In [ ]:
def classify_model(model: str) -> pd.DataFrame:
    model_paths = required_paths(model)
    n_generated = len(load_pickle_gz(model_paths["generated_structures"]))
    relaxed_ehulls = validate_1d_length(
        np.asarray(load_pickle_gz(model_paths["relaxed_ehull"]), dtype=float),
        n_generated,
        model_paths["relaxed_ehull"],
    )
    smact_validity = load_smact_validity(model_paths["smact_validity"], n_generated)
    direct = set(load_direct_matches(model_paths["direct_sm"])["gen_idx"])
    relaxed_sm_anon = load_relaxed_match_gen_indices(
        model_paths["relaxed_sm_anon_matches"]
    )
    relaxed_wyckoff = load_relaxed_match_gen_indices(
        model_paths["relaxed_wyckoff_matches"]
    )

    rows = []
    for gen_idx in range(n_generated):
        is_direct = gen_idx in direct
        has_sm_anon = gen_idx in relaxed_sm_anon
        has_wyckoff = gen_idx in relaxed_wyckoff
        ehull_relaxed = float(relaxed_ehulls[gen_idx])
        is_metastable = ehull_relaxed <= METASTABLE_EHULL_MAX
        is_smact_valid = bool(smact_validity[gen_idx])

        if is_direct:
            category = "1"
        elif has_sm_anon and has_wyckoff:
            category = "2-1"
        elif has_sm_anon:
            category = "2-2"
        elif has_wyckoff:
            category = "2-3"
        else:
            category = "3"

        rows.append(
            {
                "model": model,
                "gen_idx": gen_idx,
                "category": category,
                "category_label": CATEGORY_LABELS[category],
                "is_direct_sm_match": is_direct,
                "has_relaxed_sm_anon_match": has_sm_anon,
                "has_relaxed_wyckoff_match": has_wyckoff,
                "ehull_relaxed": ehull_relaxed,
                "is_metastable": is_metastable,
                "is_smact_valid": is_smact_valid,
                "is_metastable_smact_valid": is_metastable and is_smact_valid,
            }
        )

    frame = pd.DataFrame(rows)
    frame["category"] = pd.Categorical(
        frame["category"], categories=CATEGORY_ORDER, ordered=True
    )
    return frame


def subset_classifications(classifications: pd.DataFrame, subset: str) -> pd.DataFrame:
    if subset == "all":
        mask = pd.Series(True, index=classifications.index)
    elif subset == "metastable":
        mask = classifications["is_metastable"]
    elif subset == "metastable_smact_valid":
        mask = classifications["is_metastable_smact_valid"]
    else:
        raise ValueError(f"Unknown subset: {subset!r}")
    return classifications[mask].copy()


classifications = classify_model(MODEL)
selected_classifications = subset_classifications(classifications, SUBSET)
display(
    Markdown(
        f"**Model:** `{MODEL}`  \n"
        f"**Subset:** `{SUBSET}`  \n"
        f"**Examples per category:** `{N_EXAMPLES_PER_CATEGORY}`  \n"
        f"**Generated samples in subset:** `{len(selected_classifications):,}`"
    )
)

In [ ]:
def entries_to_frame(
    entries: list[SubstitutedEntry],
    records: list[dict[str, Any]],
    source: str,
) -> pd.DataFrame:
    if len(entries) != len(records):
        raise ValueError(
            f"{source}: entries length {len(entries)} != records length {len(records)}"
        )

    rows: list[dict[str, Any]] = []
    for entry_idx, (entry, record) in enumerate(zip(entries, records, strict=True)):
        if int(record["entry_idx"]) != entry_idx:
            raise ValueError(
                f"{source}: record entry_idx={record['entry_idx']} at position "
                f"{entry_idx}"
            )
        for attr in ("gen_idx", "train_idx", "rank"):
            if int(getattr(entry, attr)) != int(record[attr]):
                raise ValueError(f"{source}: {attr} mismatch at entry {entry_idx}")
        for attr in ("cost_uniform", "cost_mod_petti"):
            if not np.isclose(float(getattr(entry, attr)), float(record[attr])):
                raise ValueError(f"{source}: {attr} mismatch at entry {entry_idx}")

        rows.append(
            {
                "source": source,
                "source_order": SOURCE_ORDER.index(source),
                "entry_idx": entry_idx,
                "gen_idx": int(entry.gen_idx),
                "train_idx": int(entry.train_idx),
                "rank": int(entry.rank),
                "cost_uniform": float(entry.cost_uniform),
                "cost_mod_petti": float(entry.cost_mod_petti),
                "match": bool(record["match"]),
                "relaxed_substituted_structure": entry.structure,
            }
        )

    return pd.DataFrame(rows)


def load_relaxed_candidates(model_paths: dict[str, Path]) -> pd.DataFrame:
    tables = [
        entries_to_frame(
            load_pickle_gz(model_paths["relaxed_sm_anon_entries"]),
            load_pickle_gz(model_paths["relaxed_sm_anon_matches"]),
            "anon",
        ),
        entries_to_frame(
            load_pickle_gz(model_paths["relaxed_wyckoff_entries"]),
            load_pickle_gz(model_paths["relaxed_wyckoff_matches"]),
            "wyckoff",
        ),
    ]
    return pd.concat(tables, ignore_index=True)


def best_relaxed_candidates(candidates: pd.DataFrame) -> pd.DataFrame:
    matched = candidates[candidates["match"]].copy()
    if matched.empty:
        return matched
    matched = matched.sort_values(
        [
            "gen_idx",
            "cost_mod_petti",
            "cost_uniform",
            "source_order",
            "rank",
            "train_idx",
            "entry_idx",
        ],
        kind="mergesort",
    )
    return matched.drop_duplicates("gen_idx", keep="first").set_index("gen_idx")


relaxed_candidates = load_relaxed_candidates(paths)
best_candidates = best_relaxed_candidates(relaxed_candidates)
direct_train = (
    load_direct_matches(paths["direct_sm"])
    .sort_values(["gen_idx", "train_idx"], kind="mergesort")
    .drop_duplicates("gen_idx", keep="first")
    .set_index("gen_idx")["train_idx"]
)

pd.DataFrame(
    {
        "direct_match_gen_count": [len(direct_train)],
        "relaxed_match_gen_count": [len(best_candidates)],
    }
)

In [ ]:
EXAMPLE_COLUMNS = [
    "model",
    "gen_idx",
    "category",
    "category_label",
    "is_direct_sm_match",
    "has_relaxed_sm_anon_match",
    "has_relaxed_wyckoff_match",
    "ehull_relaxed",
    "is_metastable",
    "is_smact_valid",
    "is_metastable_smact_valid",
    "train_idx",
    "source",
    "rank",
    "cost_uniform",
    "cost_mod_petti",
    "relaxed_substituted_structure",
]


def select_examples(classifications: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for category in CATEGORY_ORDER:
        pool = classifications[classifications["category"] == category].sort_values(
            "gen_idx", kind="mergesort"
        )
        if category == "1":
            pool = pool[pool["gen_idx"].isin(direct_train.index)]
        elif category.startswith("2"):
            pool = pool[pool["gen_idx"].isin(best_candidates.index)]

        for _, row in pool.head(N_EXAMPLES_PER_CATEGORY).iterrows():
            gen_idx = int(row["gen_idx"])
            example = row.to_dict()
            example["train_idx"] = pd.NA
            example["source"] = pd.NA
            example["rank"] = pd.NA
            example["cost_uniform"] = pd.NA
            example["cost_mod_petti"] = pd.NA
            example["relaxed_substituted_structure"] = None

            if category == "1":
                example["train_idx"] = int(direct_train.loc[gen_idx])
            elif category.startswith("2"):
                candidate = best_candidates.loc[gen_idx]
                example["train_idx"] = int(candidate["train_idx"])
                example["source"] = str(candidate["source"])
                example["rank"] = int(candidate["rank"])
                example["cost_uniform"] = float(candidate["cost_uniform"])
                example["cost_mod_petti"] = float(candidate["cost_mod_petti"])
                example["relaxed_substituted_structure"] = candidate[
                    "relaxed_substituted_structure"
                ]

            rows.append(example)

    return pd.DataFrame(rows, columns=EXAMPLE_COLUMNS)


examples = select_examples(selected_classifications)
summary = (
    examples.groupby("category", observed=False)
    .size()
    .reindex(CATEGORY_ORDER, fill_value=0)
    .rename("selected_examples")
    .reset_index()
)
summary["category_label"] = summary["category"].map(CATEGORY_LABELS)
display(summary)
display(
    examples[
        [
            "category",
            "category_label",
            "gen_idx",
            "train_idx",
            "source",
            "rank",
            "cost_mod_petti",
            "cost_uniform",
            "ehull_relaxed",
            "is_smact_valid",
        ]
    ]
)

In [ ]:
generated_structures: list[Structure] = load_pickle_gz(paths["generated_structures"])
training_structures: list[Structure] = load_pickle_gz(paths["training_structures"])


def conventional_structure(structure: Structure) -> Structure:
    return SpacegroupAnalyzer(structure).get_conventional_standard_structure()


def structure_title(role: str, index_label: str, structure: Structure) -> str:
    analyzer = SpacegroupAnalyzer(structure)
    spg = f"{analyzer.get_space_group_symbol()} ({analyzer.get_space_group_number()})"
    formula = structure.composition.reduced_formula
    return f"{role}<br>{index_label}<br>{formula}<br>{spg}"


def format_cost(value: Any) -> str:
    if pd.isna(value):
        return "N/A"
    return f"{float(value):.6g}"


def plot_example(row: pd.Series) -> None:
    category = str(row["category"])
    gen_idx = int(row["gen_idx"])
    train_idx = None if pd.isna(row["train_idx"]) else int(row["train_idx"])

    structures: dict[str, Structure] = {
        "Generated": conventional_structure(generated_structures[gen_idx])
    }
    titles = {
        "Generated": structure_title(
            "Generated sample", f"gen_idx={gen_idx}", structures["Generated"]
        )
    }

    missing_roles = []
    if train_idx is None:
        missing_roles.append("matched train sample")
    else:
        structures["Train"] = conventional_structure(training_structures[train_idx])
        titles["Train"] = structure_title(
            "Matched train sample", f"train_idx={train_idx}", structures["Train"]
        )

    relaxed_substituted = row["relaxed_substituted_structure"]
    if relaxed_substituted is None:
        missing_roles.append("relaxed substituted train sample")
    else:
        structures["Relaxed substituted train"] = conventional_structure(
            relaxed_substituted
        )
        titles["Relaxed substituted train"] = structure_title(
            "Relaxed substituted train",
            f"{row['source']}, rank={int(row['rank'])}",
            structures["Relaxed substituted train"],
        )

    train_label = train_idx if train_idx is not None else "N/A"
    source_label = row["source"] if not pd.isna(row["source"]) else "N/A"
    missing_text = ""
    if missing_roles:
        missing_text = "  \nN/A: " + ", ".join(missing_roles)
    display(
        Markdown(
            f"### Category {category}: {CATEGORY_LABELS[category]}  \n"
            f"gen_idx=`{gen_idx}`; train_idx=`{train_label}`; "
            f"source=`{source_label}`; "
            f"mod-Pettifor cost=`{format_cost(row['cost_mod_petti'])}`; "
            f"uniform cost=`{format_cost(row['cost_uniform'])}`"
            f"{missing_text}"
        )
    )

    fig = structure_2d(
        structures,
        n_cols=3,
        show_cell=True,
        site_labels="legend",
        standardize_struct=False,
        subplot_title=lambda _struct, key: titles[key],
    )
    fig.update_layout(height=360, margin={"l": 10, "r": 10, "t": 80, "b": 10})
    display(fig)


if examples.empty:
    display(Markdown("No examples available for the selected model and subset."))
else:
    for _, example in examples.iterrows():
        plot_example(example)